<a href="https://colab.research.google.com/github/STEF-ANY/estructura-de-datos/blob/main/PROYECTO_ED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
import heapq
import math
import unicodedata



# #PARTE 1 – Grafo y datos (responsable: compañero)
# Cubre: lista de ciudades y carreteras, distancias de cada conexión, funciones de
# consulta y grafo listo para que las demás partes lo usen.


def _normalizar(texto):
    """Devuelve el texto en minúsculas, sin tildes y sin espacios sobrantes.
    Sirve para que 'bogota', 'BOGOTÁ' y ' Bogotá ' se reconozcan como la misma ciudad."""
    sin_tildes = unicodedata.normalize("NFD", texto)
    sin_tildes = "".join(c for c in sin_tildes if unicodedata.category(c) != "Mn")
    return sin_tildes.strip().lower()


class GrafoLogistico:
    """Grafo no dirigido y ponderado. Vértices = ciudades, aristas = carreteras (km)."""

    def __init__(self):
        # {ciudad: {vecino: distancia}}
        self._adyacencia = {}

    #  Construcción

    def agregar_ciudad(self, ciudad):
        """Agrega una ciudad (vértice). Si ya existe, no hace nada."""
        if not isinstance(ciudad, str) or not ciudad.strip():
            raise ValueError("El nombre de la ciudad debe ser un texto no vacío.")
        self._adyacencia.setdefault(ciudad, {})

    def agregar_carretera(self, origen, destino, distancia):
        """Agrega una carretera bidireccional. Si ya existía, actualiza su distancia.
        Valida todo ANTES de modificar el grafo para no dejar ciudades huérfanas."""
        if origen == destino:
            raise ValueError("Una carretera no puede conectar una ciudad consigo misma.")
        if (isinstance(distancia, bool) or not isinstance(distancia, (int, float))
                or not math.isfinite(distancia) or distancia <= 0):
            raise ValueError("La distancia debe ser un número positivo y finito.")
        for c in (origen, destino):
            if not isinstance(c, str) or not c.strip():
                raise ValueError("El nombre de la ciudad debe ser un texto no vacío.")

        self.agregar_ciudad(origen)
        self.agregar_ciudad(destino)
        self._adyacencia[origen][destino] = distancia
        self._adyacencia[destino][origen] = distancia

    def eliminar_carretera(self, origen, destino):
        """Elimina una carretera. Devuelve True si existía, False si no."""
        if not self.existe_carretera(origen, destino):
            return False
        del self._adyacencia[origen][destino]
        del self._adyacencia[destino][origen]
        return True

    # Consultas

    def existe_ciudad(self, ciudad):
        """True si la ciudad está en el grafo."""
        return ciudad in self._adyacencia

    def existe_carretera(self, origen, destino):
        """True si hay carretera directa entre las dos ciudades."""
        return origen in self._adyacencia and destino in self._adyacencia[origen]

    def obtener_vecinos(self, ciudad):
        """Devuelve una copia de {vecino: distancia} de la ciudad."""
        self._validar_ciudad(ciudad)
        return dict(self._adyacencia[ciudad])

    def iterar_vecinos(self, ciudad):
        """Itera pares (vecino, distancia) SIN copiar el diccionario (más eficiente
        para algoritmos como Dijkstra)."""
        self._validar_ciudad(ciudad)
        return self._adyacencia[ciudad].items()

    def obtener_distancia(self, origen, destino):
        """Distancia de la carretera directa, o None si no hay carretera directa."""
        self._validar_ciudad(origen)
        self._validar_ciudad(destino)
        return self._adyacencia[origen].get(destino)

    def grado(self, ciudad):
        """Cantidad de carreteras que salen de la ciudad."""
        self._validar_ciudad(ciudad)
        return len(self._adyacencia[ciudad])

    def listar_ciudades(self):
        """Lista de todas las ciudades."""
        return list(self._adyacencia.keys())

    def listar_carreteras(self):
        """Lista de carreteras como (origen, destino, distancia), sin repetir."""
        carreteras = []
        vistas = set()
        for origen, vecinos in self._adyacencia.items():
            for destino, distancia in vecinos.items():
                clave = frozenset((origen, destino))
                if clave not in vistas:
                    vistas.add(clave)
                    carreteras.append((origen, destino, distancia))
        return carreteras

    def obtener_grafo(self):
        """Copia completa de la lista de adyacencia."""
        return {c: dict(v) for c, v in self._adyacencia.items()}

    def a_matriz(self):
        """Devuelve (ciudades, matriz). math.inf donde no hay carretera directa y
        0 en la diagonal. Útil para Floyd-Warshall."""
        ciudades = self.listar_ciudades()
        matriz = [[0 if a == b else self._adyacencia[a].get(b, math.inf) for b in ciudades]
                  for a in ciudades]
        return ciudades, matriz

    def resolver_ciudad(self, nombre):
        """Devuelve el nombre oficial de la ciudad ignorando mayúsculas y tildes.
        Lanza KeyError si no existe. Pensado para la entrada del usuario."""
        if isinstance(nombre, str):
            buscado = _normalizar(nombre)
            for ciudad in self._adyacencia:
                if _normalizar(ciudad) == buscado:
                    return ciudad
        raise KeyError(f"La ciudad '{nombre}' no existe en el grafo.")

    # Utilidades

    def _validar_ciudad(self, ciudad):
        if ciudad not in self._adyacencia:
            raise KeyError(f"La ciudad '{ciudad}' no existe en el grafo.")

    def __len__(self):
        return len(self._adyacencia)

    def __contains__(self, ciudad):
        return ciudad in self._adyacencia

    def __str__(self):
        lineas = []
        for ciudad, vecinos in self._adyacencia.items():
            conexiones = ", ".join(f"{v} ({d} km)" for v, d in vecinos.items())
            lineas.append(f"{ciudad} -> {conexiones}")
        return "\n".join(lineas)



## 2. Datos de LogiExpress (distancias aproximadas por carretera, en km)


CIUDADES = [
    "Bogotá", "Tunja", "Villavicencio", "Girardot", "Ibagué", "Armenia",
    "Pereira", "Manizales", "Medellín", "Cali", "Popayán", "Bucaramanga",
    "Cúcuta", "Barrancabermeja", "Cartagena", "Barranquilla", "Santa Marta",
]

CARRETERAS = [
    ("Bogotá", "Tunja", 150),
    ("Bogotá", "Villavicencio", 120),
    ("Bogotá", "Girardot", 135),
    ("Bogotá", "Ibagué", 195),
    ("Bogotá", "Medellín", 415),
    ("Girardot", "Ibagué", 110),
    ("Ibagué", "Armenia", 105),
    ("Armenia", "Pereira", 45),
    ("Armenia", "Cali", 190),
    ("Pereira", "Manizales", 55),
    ("Pereira", "Medellín", 215),
    ("Manizales", "Medellín", 190),
    ("Cali", "Popayán", 140),
    ("Tunja", "Bucaramanga", 290),
    ("Bucaramanga", "Cúcuta", 200),
    ("Bucaramanga", "Barrancabermeja", 120),
    ("Barrancabermeja", "Medellín", 330),
    ("Medellín", "Cartagena", 640),
    ("Cartagena", "Barranquilla", 120),
    ("Barranquilla", "Santa Marta", 100),
]


def crear_grafo_logiexpress():
    """Construye el grafo con las ciudades y carreteras de LogiExpress."""
    grafo = GrafoLogistico()
    for ciudad in CIUDADES:
        grafo.agregar_ciudad(ciudad)
    for origen, destino, distancia in CARRETERAS:
        grafo.agregar_carretera(origen, destino, distancia)
    return grafo



# PARTE 2 – Rutas más cortas
# Cubre: algoritmo de Dijkstra, ruta más corta entre dos ciudades, costo total del
# recorrido y rutas desde un origen hacia todas las demás ciudades.
# 3. Algoritmo de Dijkstra
# Idea: se mantiene la mejor distancia conocida a cada ciudad y se procesa siempre
# la ciudad no visitada más cercana al origen (la saca un *heap*). Al procesarla se
# "relajan" sus vecinos: si pasar por ella mejora su distancia, se actualiza y se
# anota de qué ciudad se vino (predecesor) para poder reconstruir la ruta.
# Funciona porque todas las distancias son positivas.



def dijkstra(grafo, origen, destino=None):
    """Calcula las distancias mínimas desde `origen`.

    Parámetros:
        grafo:   GrafoLogistico.
        origen:  ciudad de partida (nombre exacto).
        destino: opcional. Si se indica, el algoritmo se detiene en cuanto fija
                 la distancia definitiva a esa ciudad (más rápido).

    Devuelve (distancias, predecesores):
        distancias:   {ciudad: km mínimos}; math.inf si no es alcanzable.
        predecesores: {ciudad: ciudad previa en la ruta óptima}; None para el origen
                      y para las ciudades inalcanzables.
    """
    grafo._validar_ciudad(origen)
    if destino is not None:
        grafo._validar_ciudad(destino)

    distancias = {c: math.inf for c in grafo.listar_ciudades()}
    predecesores = {c: None for c in distancias}
    distancias[origen] = 0

    cola = [(0, origen)]          # heap de (distancia acumulada, ciudad)
    visitadas = set()

    while cola:
        dist_actual, ciudad = heapq.heappop(cola)

        if ciudad in visitadas:   # entrada obsoleta: ya se procesó con mejor distancia
            continue
        visitadas.add(ciudad)

        if ciudad == destino:     # distancia al destino ya es definitiva
            break

        for vecino, peso in grafo.iterar_vecinos(ciudad):
            if vecino in visitadas:
                continue
            nueva = dist_actual + peso
            if nueva < distancias[vecino]:      # relajación de la arista
                distancias[vecino] = nueva
                predecesores[vecino] = ciudad
                heapq.heappush(cola, (nueva, vecino))

    return distancias, predecesores


def reconstruir_ruta(predecesores, origen, destino):
    """Reconstruye la lista de ciudades origen -> destino a partir de los
    predecesores. Devuelve None si el destino no es alcanzable."""
    ruta = []
    actual = destino
    while actual is not None:
        ruta.append(actual)
        actual = predecesores[actual]
    ruta.reverse()
    return ruta if ruta and ruta[0] == origen else None



# 4. Funcionalidades pedidas

def ruta_mas_corta(grafo, origen, destino):
    """Ruta más corta entre dos ciudades.

    Devuelve (ruta, costo_total_km). Si no hay camino: (None, math.inf).
    Acepta nombres sin importar mayúsculas/tildes ('bogota' -> 'Bogotá')."""
    origen = grafo.resolver_ciudad(origen)
    destino = grafo.resolver_ciudad(destino)
    distancias, predecesores = dijkstra(grafo, origen, destino)
    ruta = reconstruir_ruta(predecesores, origen, destino)
    if ruta is None:
        return None, math.inf
    return ruta, distancias[destino]


def costo_total_ruta(grafo, ruta):
    """Suma las distancias de las carreteras de una ruta (lista de ciudades).
    Lanza ValueError si dos ciudades consecutivas no tienen carretera directa."""
    total = 0
    for a, b in zip(ruta, ruta[1:]):
        if not grafo.existe_carretera(a, b):
            raise ValueError(f"No hay carretera directa entre {a} y {b}.")
        total += grafo.obtener_distancia(a, b)
    return total


def rutas_desde_origen(grafo, origen):
    """Rutas más cortas desde `origen` hacia TODAS las demás ciudades.

    Devuelve {ciudad: (ruta, costo_km)}, ordenado de la más cercana a la más
    lejana. Las ciudades inalcanzables tienen (None, math.inf)."""
    origen = grafo.resolver_ciudad(origen)
    distancias, predecesores = dijkstra(grafo, origen)
    resultado = {}
    for ciudad in sorted(distancias, key=distancias.get):
        if ciudad == origen:
            continue
        ruta = reconstruir_ruta(predecesores, origen, ciudad)
        resultado[ciudad] = (ruta, distancias[ciudad]) if ruta else (None, math.inf)
    return resultado


def ruta_multipunto(grafo, puntos):
    """Ruta que pasa por varias ciudades en el orden dado (origen, paradas..., destino).

    Une la ruta más corta de cada tramo. Devuelve (ruta_completa, costo_total).
    Si algún tramo es imposible: (None, math.inf)."""
    if len(puntos) < 2:
        raise ValueError("Se necesitan al menos un origen y un destino.")
    completa, total = [], 0
    for a, b in zip(puntos, puntos[1:]):
        tramo, costo = ruta_mas_corta(grafo, a, b)
        if tramo is None:
            return None, math.inf
        completa.extend(tramo if not completa else tramo[1:])  # evita repetir la unión
        total += costo
    return completa, total



#  5. Pruebas (Parte 1 y Parte 2)

if __name__ == "__main__":
    grafo = crear_grafo_logiexpress()
    print(f"Ciudades: {len(grafo)} | Carreteras: {len(grafo.listar_carreteras())}\n")

    #  Pruebas de la PARTE 1 (grafo)
    print("Vecinos de Bogotá:", grafo.obtener_vecinos("Bogotá"))
    print("Distancia Bogotá-Tunja:", grafo.obtener_distancia("Bogotá", "Tunja"), "km")
    hay = grafo.existe_carretera("Bogotá", "Cali")
    print("¿Hay carretera directa Bogotá-Cali?", "Sí" if hay else "No")
    # El grado de una ciudad es la cantidad de carreteras directas que salen de ella
    grado = grafo.grado("Medellín")
    vecinos = ", ".join(grafo.obtener_vecinos("Medellín"))
    print(f"Medellín tiene {grado} carreteras directas (grado = {grado}), hacia: {vecinos}\n")

    #  Pruebas de la PARTE 2 (Dijkstra)
    # Ruta más corta + costo total
    ruta, costo = ruta_mas_corta(grafo, "bogota", "Cali")
    print("Bogotá -> Cali:", " -> ".join(ruta), f"({costo} km)")
    assert costo == costo_total_ruta(grafo, ruta)   # el costo coincide con la suma de tramos

    # Caso donde hay dos caminos posibles (directo vs. por el Eje Cafetero)
    ruta, costo = ruta_mas_corta(grafo, "Bogotá", "Medellín")
    print("Bogotá -> Medellín:", " -> ".join(ruta), f"({costo} km)")

    # Origen = destino
    ruta, costo = ruta_mas_corta(grafo, "Bogotá", "Bogotá")
    print(f"Bogotá -> Bogotá: ya estás en el destino, ruta {ruta} y costo {costo} km")

    # Todas las rutas desde un origen
    print("\nRutas desde Bogotá:")
    for ciudad, (r, c) in rutas_desde_origen(grafo, "Bogotá").items():
        print(f"  {ciudad:<16} {c:>5} km   {' -> '.join(r)}")

    # Ruta con paradas
    r, c = ruta_multipunto(grafo, ["Bogotá", "Medellín", "Santa Marta"])
    print("\nMultipunto:", " -> ".join(r), f"({c} km)")

    # Ciudad inalcanzable (se agrega una ciudad aislada)
    grafo.agregar_ciudad("Leticia")
    ruta, costo = ruta_mas_corta(grafo, "Bogotá", "Leticia")
    if ruta is None:
        print("Bogotá -> Leticia: no hay ruta posible (Leticia no tiene carreteras conectadas)")

    # Ciudad inexistente
    try:
        ruta_mas_corta(grafo, "Bogotá", "Atlantis")
    except KeyError as e:
        print("Bogotá -> Atlantis: ciudad inexistente, error controlado:", e)

Ciudades: 17 | Carreteras: 20

Vecinos de Bogotá: {'Tunja': 150, 'Villavicencio': 120, 'Girardot': 135, 'Ibagué': 195, 'Medellín': 415}
Distancia Bogotá-Tunja: 150 km
¿Hay carretera directa Bogotá-Cali? No
Medellín tiene 5 carreteras directas (grado = 5), hacia: Bogotá, Pereira, Manizales, Barrancabermeja, Cartagena

Bogotá -> Cali: Bogotá -> Ibagué -> Armenia -> Cali (490 km)
Bogotá -> Medellín: Bogotá -> Medellín (415 km)
Bogotá -> Bogotá: ya estás en el destino, ruta ['Bogotá'] y costo 0 km

Rutas desde Bogotá:
  Villavicencio      120 km   Bogotá -> Villavicencio
  Girardot           135 km   Bogotá -> Girardot
  Tunja              150 km   Bogotá -> Tunja
  Ibagué             195 km   Bogotá -> Ibagué
  Armenia            300 km   Bogotá -> Ibagué -> Armenia
  Pereira            345 km   Bogotá -> Ibagué -> Armenia -> Pereira
  Manizales          400 km   Bogotá -> Ibagué -> Armenia -> Pereira -> Manizales
  Medellín           415 km   Bogotá -> Medellín
  Bucaramanga        440 k

In [ ]:
class GrafoLogistico:
    def __init__(self):
        self._adyacencia = {}

    # Construcción

    def agregar_ciudad(self, ciudad):
        if not isinstance(ciudad, str) or not ciudad.strip():
            raise ValueError("El nombre de la ciudad debe ser un texto no vacío.")
        self._adyacencia.setdefault(ciudad, {})

    def agregar_carretera(self, origen, destino, distancia):
        if origen == destino:
            raise ValueError("Una carretera no puede conectar una ciudad consigo misma.")
        if not isinstance(distancia, (int, float)) or distancia <= 0:
            raise ValueError("La distancia debe ser un número positivo.")

        self.agregar_ciudad(origen)
        self.agregar_ciudad(destino)
        self._adyacencia[origen][destino] = distancia
        self._adyacencia[destino][origen] = distancia

    def eliminar_carretera(self, origen, destino):
        if not self.existe_carretera(origen, destino):
            return False
        del self._adyacencia[origen][destino]
        del self._adyacencia[destino][origen]
        return True

    # Consultas

    def existe_ciudad(self, ciudad):
        return ciudad in self._adyacencia

    def existe_carretera(self, origen, destino):
        return origen in self._adyacencia and destino in self._adyacencia[origen]

    def obtener_vecinos(self, ciudad):
        self._validar_ciudad(ciudad)
        return dict(self._adyacencia[ciudad])

    def obtener_distancia(self, origen, destino):
        self._validar_ciudad(origen)
        self._validar_ciudad(destino)
        return self._adyacencia[origen].get(destino)

    def grado(self, ciudad):
        self._validar_ciudad(ciudad)
        return len(self._adyacencia[ciudad])

    def listar_ciudades(self):
        return list(self._adyacencia.keys())

    def listar_carreteras(self):
        carreteras = []
        vistas = set()
        for origen, vecinos in self._adyacencia.items():
            for destino, distancia in vecinos.items():
                clave = frozenset((origen, destino))
                if clave not in vistas:
                    vistas.add(clave)
                    carreteras.append((origen, destino, distancia))
        return carreteras

    def obtener_grafo(self):
        return {c: dict(v) for c, v in self._adyacencia.items()}


    # Utilidades

    def _validar_ciudad(self, ciudad):
        if ciudad not in self._adyacencia:
            raise KeyError(f"La ciudad '{ciudad}' no existe en el grafo.")

    def __len__(self):
        return len(self._adyacencia)

    def __contains__(self, ciudad):
        return ciudad in self._adyacencia

    def __str__(self):
        lineas = []
        for ciudad, vecinos in self._adyacencia.items():
            conexiones = ", ".join(f"{v} ({d} km)" for v, d in vecinos.items())
            lineas.append(f"{ciudad} -> {conexiones}")
        return "\n".join(lineas)


# Datos de LogiExpress (distancias aproximadas por carretera, en km)

CIUDADES = [
    "Bogotá", "Tunja", "Villavicencio", "Girardot", "Ibagué", "Armenia",
    "Pereira", "Manizales", "Medellín", "Cali", "Popayán", "Bucaramanga",
    "Cúcuta", "Barrancabermeja", "Cartagena", "Barranquilla", "Santa Marta",
]

CARRETERAS = [
    ("Bogotá", "Tunja", 150),
    ("Bogotá", "Villavicencio", 120),
    ("Bogotá", "Girardot", 135),
    ("Bogotá", "Ibagué", 195),
    ("Bogotá", "Medellín", 415),
    ("Girardot", "Ibagué", 110),
    ("Ibagué", "Armenia", 105),
    ("Armenia", "Pereira", 45),
    ("Armenia", "Cali", 190),
    ("Pereira", "Manizales", 55),
    ("Pereira", "Medellín", 215),
    ("Manizales", "Medellín", 190),
    ("Cali", "Popayán", 140),
    ("Tunja", "Bucaramanga", 290),
    ("Bucaramanga", "Cúcuta", 200),
    ("Bucaramanga", "Barrancabermeja", 120),
    ("Barrancabermeja", "Medellín", 330),
    ("Medellín", "Cartagena", 640),
    ("Cartagena", "Barranquilla", 120),
    ("Barranquilla", "Santa Marta", 100)
]


def crear_grafo_logiexpress():
    grafo = GrafoLogistico()
    for ciudad in CIUDADES:
        grafo.agregar_ciudad(ciudad)
    for origen, destino, distancia in CARRETERAS:
        grafo.agregar_carretera(origen, destino, distancia)
    return grafo



# Prueba
if __name__ == "__main__":
    grafo = crear_grafo_logiexpress()

    print(f"Ciudades: {len(grafo)}")
    print(f"Carreteras: {len(grafo.listar_carreteras())}\n")
    print(grafo, "\n")

    print("Vecinos de Bogotá:", grafo.obtener_vecinos("Bogotá"))
    print("Distancia Bogotá-Tunja:", grafo.obtener_distancia("Bogotá", "Tunja"), "km")
    print("¿Hay carretera directa Bogotá-Cali?", grafo.existe_carretera("Bogotá", "Cali"))
    print("Grado de Medellín:", grafo.grado("Medellín"))



Ciudades: 17
Carreteras: 20

Bogotá -> Tunja (150 km), Villavicencio (120 km), Girardot (135 km), Ibagué (195 km), Medellín (415 km)
Tunja -> Bogotá (150 km), Bucaramanga (290 km)
Villavicencio -> Bogotá (120 km)
Girardot -> Bogotá (135 km), Ibagué (110 km)
Ibagué -> Bogotá (195 km), Girardot (110 km), Armenia (105 km)
Armenia -> Ibagué (105 km), Pereira (45 km), Cali (190 km)
Pereira -> Armenia (45 km), Manizales (55 km), Medellín (215 km)
Manizales -> Pereira (55 km), Medellín (190 km)
Medellín -> Bogotá (415 km), Pereira (215 km), Manizales (190 km), Barrancabermeja (330 km), Cartagena (640 km)
Cali -> Armenia (190 km), Popayán (140 km)
Popayán -> Cali (140 km)
Bucaramanga -> Tunja (290 km), Cúcuta (200 km), Barrancabermeja (120 km)
Cúcuta -> Bucaramanga (200 km)
Barrancabermeja -> Bucaramanga (120 km), Medellín (330 km)
Cartagena -> Medellín (640 km), Barranquilla (120 km)
Barranquilla -> Cartagena (120 km), Santa Marta (100 km)
Santa Marta -> Barranquilla (100 km) 

Vecinos de Bog

In [ ]:
# ==============================================================================
# PARTE 3: INTERFAZ GRAFICA Y VISUALIZACION DE RUTAS
# PROYECTO: LOGIEXPRESS
# ==============================================================================

import folium
import requests
import ipywidgets as widgets
import json
import heapq

from branca.element import Element, MacroElement, Template
from IPython.display import display, clear_output


# ==============================================================================
# IDENTIDAD VISUAL LOGIEXPRESS
# ==============================================================================

# Paleta fluorescente sobre fondo gris azulado.
FONDO = "#263746"
PANEL = "#34495A"
PANEL_CLARO = "#40576A"
CIAN = "#00E5FF"
FUCSIA = "#FF35C8"
VIOLETA = "#B56BFF"
VERDE = "#B7FF36"
BLANCO = "#F3F7FA"
GRIS_TEXTO = "#C1CDD6"

AZUL = CIAN
ROSADO = FUCSIA
COLORES_RUTAS = [CIAN, FUCSIA, VIOLETA]


# ==============================================================================
# COORDENADAS GEOGRAFICAS
# ==============================================================================

COORDENADAS_CIUDADES = {
    "Bogotá": [4.7110, -74.0721],
    "Tunja": [5.5353, -73.3678],
    "Villavicencio": [4.1420, -73.6266],
    "Girardot": [4.3039, -74.8031],
    "Ibagué": [4.4389, -75.2322],
    "Armenia": [4.5339, -75.6811],
    "Pereira": [4.8087, -75.6906],
    "Manizales": [5.0703, -75.5138],
    "Medellín": [6.2476, -75.5658],
    "Cali": [3.4516, -76.5320],
    "Popayán": [2.4448, -76.6147],
    "Bucaramanga": [7.1193, -73.1227],
    "Cúcuta": [7.8891, -72.4967],
    "Barrancabermeja": [7.0653, -73.8547],
    "Cartagena": [10.3910, -75.4794],
    "Barranquilla": [10.9685, -74.7813],
    "Santa Marta": [11.2408, -74.1990]
}


# ==============================================================================
# CONFIGURACION DEL MAPA
# ==============================================================================

def crear_mapa(ciudades_mostrar):
    """
    Construye un mapa interactivo de Colombia.

    Parámetros:
        ciudades_mostrar: ciudades que deben quedar visibles.

    Retorna:
        Mapa Folium preparado para incorporar rutas y marcadores.
    """

    coordenadas = [
        COORDENADAS_CIUDADES[ciudad]
        for ciudad in ciudades_mostrar
        if ciudad in COORDENADAS_CIUDADES
    ]

    if not coordenadas:
        coordenadas = [[4.7110, -74.0721]]

    mapa = folium.Map(
        location=coordenadas[0],
        zoom_start=6,
        tiles=None,
        control_scale=True
    )

    # Mapa de carreteras claro para facilitar la lectura
    # de las rutas fluorescentes.
    folium.TileLayer(
        tiles=(
            "https://server.arcgisonline.com/ArcGIS/rest/services/"
            "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
        ),
        attr="Esri",
        name="Mapa de carreteras",
        overlay=False,
        control=True
    ).add_to(mapa)

    folium.LayerControl(collapsed=True).add_to(mapa)

    mapa.fit_bounds(coordenadas)

    return mapa


# ==============================================================================
# DIJKSTRA CON RESTRICCIONES
# ==============================================================================

def camino_minimo_restringido(
    grafo,
    origen,
    destino,
    nodos_prohibidos=None,
    aristas_prohibidas=None
):
    """
    Calcula el camino de menor costo mediante Dijkstra.

    Permite excluir temporalmente ciudades y conexiones para
    generar rutas alternativas.

    Retorna:
        Tupla (ruta, costo) o None si no existe un camino.
    """

    nodos_prohibidos = nodos_prohibidos or set()
    aristas_prohibidas = aristas_prohibidas or set()

    if origen in nodos_prohibidos or destino in nodos_prohibidos:
        return None

    cola = [(0, origen, (origen,))]
    visitadas = set()

    while cola:

        costo, ciudad, ruta = heapq.heappop(cola)

        if ciudad in visitadas:
            continue

        visitadas.add(ciudad)

        if ciudad == destino:
            return list(ruta), costo

        for vecino, distancia in grafo.iterar_vecinos(ciudad):

            if vecino in visitadas:
                continue

            if vecino in nodos_prohibidos:
                continue

            arista = frozenset((ciudad, vecino))

            if arista in aristas_prohibidas:
                continue

            heapq.heappush(
                cola,
                (
                    costo + distancia,
                    vecino,
                    ruta + (vecino,)
                )
            )

    return None


# ==============================================================================
# GENERACION DE RUTAS ALTERNATIVAS - METODO DE YEN
# ==============================================================================

def calcular_varias_rutas(grafo, origen, destino, cantidad=3):
    """
    Calcula hasta tres rutas alternativas mediante el método de Yen.

    Utiliza Dijkstra para encontrar los caminos de menor costo,
    evitando repetir rutas ya aceptadas.

    Retorna:
        Lista de tuplas (ruta, costo), ordenadas por costo.
    """

    origen = grafo.resolver_ciudad(origen)
    destino = grafo.resolver_ciudad(destino)

    if origen == destino:
        return []

    primera = camino_minimo_restringido(grafo, origen, destino)

    if primera is None:
        return []

    rutas_aceptadas = [primera]
    candidatos = {}

    for _ in range(cantidad - 1):

        ruta_anterior = rutas_aceptadas[-1][0]

        for i in range(len(ruta_anterior) - 1):

            raiz = ruta_anterior[:i + 1]
            ciudad_desvio = raiz[-1]

            aristas_prohibidas = set()

            for ruta, costo in rutas_aceptadas:

                if ruta[:i + 1] == raiz:
                    aristas_prohibidas.add(
                        frozenset((ruta[i], ruta[i + 1]))
                    )

            nodos_prohibidos = set(raiz[:-1])

            desvio = camino_minimo_restringido(
                grafo,
                ciudad_desvio,
                destino,
                nodos_prohibidos,
                aristas_prohibidas
            )

            if desvio is None:
                continue

            ruta_completa = raiz[:-1] + desvio[0]
            clave = tuple(ruta_completa)

            if any(
                tuple(ruta) == clave
                for ruta, costo in rutas_aceptadas
            ):
                continue

            if clave in candidatos:
                continue

            costo = costo_total_ruta(grafo, ruta_completa)
            candidatos[clave] = costo

        if not candidatos:
            break

        mejor_ruta = min(
            candidatos,
            key=lambda ruta: candidatos[ruta]
        )

        mejor_costo = candidatos.pop(mejor_ruta)

        rutas_aceptadas.append(
            (list(mejor_ruta), mejor_costo)
        )

    return rutas_aceptadas


# ==============================================================================
# CONSULTA DE RECORRIDOS VIALES CON OSRM
# ==============================================================================

def obtener_recorrido_vial(ruta):
    """
    Consulta OSRM para obtener el trazado vial de una ruta.

    Parámetros:
        ruta: lista ordenada de ciudades.

    Retorna:
        Coordenadas del recorrido y distancia vial en kilómetros.
    """

    if not ruta or len(ruta) < 2:
        raise ValueError("La ruta debe tener al menos dos ciudades.")

    coordenadas_osrm = ";".join(
        f"{COORDENADAS_CIUDADES[ciudad][1]},"
        f"{COORDENADAS_CIUDADES[ciudad][0]}"
        for ciudad in ruta
    )

    url = (
        "https://router.project-osrm.org/route/v1/driving/"
        + coordenadas_osrm
    )

    parametros = {
        "overview": "full",
        "geometries": "geojson",
        "steps": "false"
    }

    respuesta = requests.get(
        url,
        params=parametros,
        timeout=30
    )

    respuesta.raise_for_status()

    datos = respuesta.json()

    if datos.get("code") != "Ok" or not datos.get("routes"):
        raise ValueError("No se encontró un recorrido vial.")

    coordenadas = datos["routes"][0]["geometry"]["coordinates"]

    recorrido = [
        [latitud, longitud]
        for longitud, latitud in coordenadas
    ]

    distancia_km = datos["routes"][0]["distance"] / 1000

    return recorrido, distancia_km


# ==============================================================================
# CONFIGURACION DEL GRAFO
# ==============================================================================

grafo_interfaz = crear_grafo_logiexpress()

ciudades = sorted(grafo_interfaz._adyacencia.keys())


# ==============================================================================
# ELEMENTOS DE LA INTERFAZ
# ==============================================================================

selector_origen = widgets.Dropdown(
    options=ciudades,
    layout=widgets.Layout(width="100%", height="42px")
)

selector_destino = widgets.Dropdown(
    options=ciudades,
    layout=widgets.Layout(width="100%", height="42px")
)

selector_parada = widgets.Dropdown(
    options=ciudades,
    layout=widgets.Layout(width="100%", height="42px")
)

paradas_seleccionadas = []

boton_agregar_parada = widgets.Button(
    description="＋ AÑADIR PARADA",
    layout=widgets.Layout(width="175px", height="40px")
)

boton_calcular = widgets.Button(
    description="CALCULAR RECORRIDO  →",
    layout=widgets.Layout(width="100%", height="48px")
)

boton_consultar = widgets.Button(
    description="EXPLORAR ALTERNATIVAS",
    layout=widgets.Layout(width="100%", height="48px")
)

salida_interfaz = widgets.Output()
contenedor_paradas = widgets.VBox()
linea_recorrido = widgets.HTML()


# ==============================================================================
# ACTUALIZACION DE LA SECUENCIA VISUAL
# ==============================================================================

def actualizar_recorrido_visual():
    """
    Actualiza la secuencia gráfica de origen, paradas y destino.
    """

    puntos = (
        [selector_origen.value]
        + paradas_seleccionadas
        + [selector_destino.value]
    )

    html = """
    <div class="ruta-visual">
        <div class="ruta-titulo">SECUENCIA DEL RECORRIDO</div>
        <div class="ruta-linea">
    """

    for i, ciudad in enumerate(puntos):

        if i == 0:
            clase = "origen"
            etiqueta = "INICIO"

        elif i == len(puntos) - 1:
            clase = "destino"
            etiqueta = "LLEGADA"

        else:
            clase = "parada"
            etiqueta = f"PARADA {i}"

        html += f"""
        <div class="ruta-punto">
            <div class="ruta-circulo {clase}">{i + 1}</div>
            <div class="ruta-ciudad">{ciudad}</div>
            <div class="ruta-etiqueta">{etiqueta}</div>
        </div>
        """

        if i < len(puntos) - 1:
            html += '<div class="ruta-conector"></div>'

    html += """
        </div>
    </div>
    """

    linea_recorrido.value = html


# ==============================================================================
# ADMINISTRACION DE PARADAS
# ==============================================================================

def actualizar_lista_paradas():
    """
    Actualiza el listado visual de ciudades intermedias.
    """

    elementos = []

    if not paradas_seleccionadas:

        elementos.append(
            widgets.HTML(
                "<div class='sin-paradas'>"
                "NO HAY PARADAS INTERMEDIAS REGISTRADAS"
                "</div>"
            )
        )

    for ciudad in paradas_seleccionadas:

        boton_quitar = widgets.Button(
            description="✕",
            tooltip=f"Quitar {ciudad}",
            layout=widgets.Layout(width="38px", height="34px")
        )

        boton_quitar.add_class("boton-quitar")

        def quitar_parada(_, ciudad_actual=ciudad):

            if ciudad_actual in paradas_seleccionadas:
                paradas_seleccionadas.remove(ciudad_actual)

            actualizar_lista_paradas()
            actualizar_recorrido_visual()

        boton_quitar.on_click(quitar_parada)

        fila = widgets.HBox(
            [
                widgets.HTML(
                    f"<div class='parada-item'>"
                    f"<span class='punto-mini'></span>{ciudad}</div>"
                ),
                boton_quitar
            ],
            layout=widgets.Layout(
                width="100%",
                justify_content="space-between",
                align_items="center"
            )
        )

        elementos.append(fila)

    contenedor_paradas.children = tuple(elementos)


def agregar_parada(_):
    """
    Agrega una ciudad como parada intermedia, evitando duplicados.
    """

    ciudad = selector_parada.value

    if ciudad in [selector_origen.value, selector_destino.value]:

        with salida_interfaz:
            clear_output(wait=True)
            print("Esa ciudad ya está seleccionada como origen o destino.")

        return

    if ciudad in paradas_seleccionadas:

        with salida_interfaz:
            clear_output(wait=True)
            print("Esa parada ya está agregada.")

        return

    paradas_seleccionadas.append(ciudad)

    actualizar_lista_paradas()
    actualizar_recorrido_visual()

    with salida_interfaz:
        clear_output(wait=True)
        print(f"✓ Parada agregada: {ciudad}")


def cambiar_ciudades(change):
    """
    Elimina paradas que coincidan con el nuevo origen o destino.
    """

    if change["name"] == "value":

        for ciudad in paradas_seleccionadas[:]:

            if ciudad in [
                selector_origen.value,
                selector_destino.value
            ]:
                paradas_seleccionadas.remove(ciudad)

        actualizar_lista_paradas()
        actualizar_recorrido_visual()


boton_agregar_parada.on_click(agregar_parada)

selector_origen.observe(cambiar_ciudades, names="value")
selector_destino.observe(cambiar_ciudades, names="value")

actualizar_lista_paradas()
actualizar_recorrido_visual()


# ==============================================================================
# MARCADORES DEL MAPA
# ==============================================================================

def agregar_marcadores_ruta(mapa, ruta):
    """
    Agrega marcadores diferenciados para origen, destino e intermedios.
    """

    for i, ciudad in enumerate(ruta):

        if i == 0:
            color = "green"
            tipo = "Origen"

        elif i == len(ruta) - 1:
            color = "red"
            tipo = "Destino"

        else:
            color = "purple"
            tipo = "Ciudad intermedia"

        folium.Marker(
            location=COORDENADAS_CIUDADES[ciudad],
            tooltip=f"{tipo}: {ciudad}",
            popup=f"{tipo}: {ciudad}",
            icon=folium.Icon(
                color=color,
                icon="info-sign"
            )
        ).add_to(mapa)


# ==============================================================================
# VISUALIZACION DE UNA RUTA
# ==============================================================================

def visualizar_grafo(ruta=None):
    """
    Dibuja una ruta en el mapa y muestra su distancia vial.
    """

    if not ruta or len(ruta) < 2:
        print("No hay una ruta disponible para visualizar.")
        return None

    mapa = crear_mapa(ruta)

    try:

        recorrido, distancia_km = obtener_recorrido_vial(ruta)

        agregar_marcadores_ruta(mapa, ruta)

        # Línea principal cian fluorescente.
        folium.PolyLine(
            locations=recorrido,
            color=CIAN,
            weight=7,
            opacity=1,
            tooltip="Ruta recomendada"
        ).add_to(mapa)

        mapa.fit_bounds(recorrido)

        print("RECORRIDO POR CARRETERAS")
        print(f"Distancia vial OSRM: {distancia_km:.2f} km")
        print("RUTA CALCULADA POR DIJKSTRA:")
        print(" → ".join(ruta))

        return mapa

    except Exception as error:

        print("No fue posible consultar el trazado vial.")
        print(f"Detalle: {error}")

        return None


# ==============================================================================
# CALCULO DE RUTA CON PARADAS INTERMEDIAS
# ==============================================================================

def calcular_ruta_interfaz(_):
    """
    Calcula la ruta que conecta origen, paradas y destino.
    """

    with salida_interfaz:

        clear_output(wait=True)

        origen = selector_origen.value
        destino = selector_destino.value

        if origen == destino:
            print("El origen y el destino deben ser diferentes.")
            return

        intermedios = list(paradas_seleccionadas)

        if origen in intermedios or destino in intermedios:
            print("El origen y el destino no pueden repetirse como paradas.")
            return

        puntos_recorrido = [origen] + intermedios + [destino]

        try:

            ruta, costo_total = ruta_multipunto(
                grafo_interfaz,
                puntos_recorrido
            )

            if not ruta:
                print("No se encontró una ruta disponible.")
                return

            print("RUTA RECOMENDADA")
            print(" → ".join(ruta))
            print(f"Número de tramos: {len(ruta) - 1}")
            print(f"Costo total del grafo: {costo_total} km")

            mapa_resultado = visualizar_grafo(ruta)

            if mapa_resultado is not None:
                display(mapa_resultado)

        except (ValueError, KeyError) as error:

            print(f"No fue posible calcular la ruta: {error}")


# ==============================================================================
# RUTAS ALTERNATIVAS
# ==============================================================================

def consultar_rutas_interfaz(_):
    """
    Calcula y dibuja rutas alternativas entre origen y destino.

    Permite seleccionar una ruta haciendo clic sobre su trazado.
    """

    with salida_interfaz:

        clear_output(wait=True)

        origen = selector_origen.value
        destino = selector_destino.value

        if origen == destino:
            print("El origen y el destino deben ser diferentes.")
            return

        try:

            resultados = calcular_varias_rutas(
                grafo_interfaz,
                origen,
                destino,
                cantidad=3
            )

            if not resultados:

                print(
                    f"No hay rutas disponibles entre {origen} "
                    f"y {destino} en el grafo."
                )
                return

            mapa = crear_mapa(
                [
                    ciudad
                    for ruta, costo in resultados
                    for ciudad in ruta
                ]
            )

            folium.Marker(
                location=COORDENADAS_CIUDADES[origen],
                tooltip=f"Origen: {origen}",
                icon=folium.Icon(color="green", icon="play")
            ).add_to(mapa)

            folium.Marker(
                location=COORDENADAS_CIUDADES[destino],
                tooltip=f"Destino: {destino}",
                icon=folium.Icon(color="red", icon="flag")
            ).add_to(mapa)

            capas_rutas = []
            errores = []

            print(f"ALTERNATIVAS: {origen} → {destino}")
            print("Consultando los trazados viales...")

            for numero, (ruta, costo) in enumerate(resultados, start=1):

                try:

                    recorrido, distancia_osrm = obtener_recorrido_vial(ruta)

                    color_ruta = COLORES_RUTAS[
                        (numero - 1) % len(COLORES_RUTAS)
                    ]

                    capa = folium.PolyLine(
                        locations=recorrido,
                        color=color_ruta,
                        weight=6,
                        opacity=0.95,
                        tooltip=f"Ruta {numero} - {distancia_osrm:.2f} km"
                    )

                    capa.add_to(mapa)

                    capas_rutas.append({
                        "capa": capa,
                        "numero": numero,
                        "ruta": ruta,
                        "costo": costo,
                        "distancia": round(distancia_osrm, 2),
                        "color": color_ruta
                    })

                except Exception as error:

                    errores.append(f"Ruta {numero}: {str(error)}")

            if not capas_rutas:

                print("No se pudieron dibujar las rutas en el mapa.")

                for error in errores:
                    print("-", error)

                return

            todas_coordenadas = []

            for elemento in capas_rutas:
                todas_coordenadas.extend(elemento["capa"].locations)

            mapa.fit_bounds(todas_coordenadas)

            # Panel de información con fondo grafito azulado.
            panel = """
            <div id="informacion-ruta" style="
                position:fixed;
                bottom:25px;
                left:50px;
                z-index:9999;
                background:#34495A;
                color:#F3F7FA;
                padding:17px;
                border:1px solid #00E5FF;
                border-left:5px solid #FF35C8;
                border-radius:10px;
                font-family:Arial;
                font-size:13px;
                max-width:410px;
                box-shadow:0 0 18px rgba(0,229,255,.28);">
                <b style="color:#00E5FF">LOGIEXPRESS</b><br>
                Selecciona una ruta para consultar sus detalles.
            </div>
            """

            instruccion = """
            <div id="instruccion-ruta" style="
                position:fixed;
                top:12px;
                left:50px;
                z-index:9999;
                background:#34495A;
                color:#F3F7FA;
                padding:13px 17px;
                border:1px solid #00E5FF;
                border-bottom:3px solid #FF35C8;
                border-radius:9px;
                font-family:Arial;
                font-size:13px;
                box-shadow:0 0 15px rgba(0,229,255,.22);">
                <b style="color:#00E5FF">RUTAS ALTERNATIVAS</b><br>
                Selecciona una línea para ver sus detalles.
            </div>
            """

            mapa.get_root().html.add_child(Element(panel))
            mapa.get_root().html.add_child(Element(instruccion))

            entradas_js = []

            for elemento in capas_rutas:

                entradas_js.append(
                    "{"
                    f"capa: {elemento['capa'].get_name()},"
                    f"numero: {elemento['numero']},"
                    f"ruta: {json.dumps(elemento['ruta'], ensure_ascii=False)},"
                    f"costo: {json.dumps(elemento['costo'])},"
                    f"distancia: {json.dumps(elemento['distancia'])},"
                    f"color: {json.dumps(elemento['color'])}"
                    "}"
                )

            lista_rutas_js = ",".join(entradas_js)

            macro = MacroElement()

            macro._template = Template("""
            {% macro script(this, kwargs) %}

            var mapaSeleccion = {{ this.mapa }};
            var rutasDisponibles = [{{ this.rutas }}];
            var marcadoresSeleccionados = [];
            var rutaActiva = null;

            // Permite seleccionar una ruta y volver a cambiar de alternativa.
            rutasDisponibles.forEach(function(elemento) {

                elemento.capa.on('click', function() {

                    // Elimina los marcadores de la selección anterior.
                    marcadoresSeleccionados.forEach(function(marcador) {
                        mapaSeleccion.removeLayer(marcador);
                    });

                    marcadoresSeleccionados = [];

                    // Restaura el estilo y la visibilidad de todas las rutas.
                    rutasDisponibles.forEach(function(otra) {

                        if (!mapaSeleccion.hasLayer(otra.capa)) {
                            otra.capa.addTo(mapaSeleccion);
                        }

                        otra.capa.setStyle({
                            color: otra.color,
                            weight: 6,
                            opacity: 0.95
                        });
                    });

                    rutaActiva = elemento;

                    // Destaca la alternativa seleccionada.
                    elemento.capa.setStyle({
                        color: '#B7FF36',
                        weight: 9,
                        opacity: 1
                    });

                    elemento.capa.bringToFront();

                    // Añade marcadores de las ciudades de esa ruta.
                    elemento.ruta.forEach(function(ciudad, indice) {

                        var coordenadas = null;

                        switch(ciudad) {

                            {% for ciudad, coordenada in this.coordenadas.items() %}
                            case {{ ciudad | tojson }}:
                                coordenadas = {{ coordenada | tojson }};
                                break;
                            {% endfor %}

                        }

                        if (coordenadas !== null) {

                            var color = '#B56BFF';
                            var tipo = 'Intermedia';

                            if (indice === 0) {
                                color = '#B7FF36';
                                tipo = 'Origen';
                            }

                            if (indice === elemento.ruta.length - 1) {
                                color = '#FF35C8';
                                tipo = 'Destino';
                            }

                            var marcador = L.circleMarker(coordenadas, {
                                radius: 9,
                                color: color,
                                fillColor: color,
                                fillOpacity: 1,
                                weight: 3
                            }).bindTooltip(tipo + ': ' + ciudad);

                            marcador.addTo(mapaSeleccion);
                            marcadoresSeleccionados.push(marcador);
                        }
                    });

                    // Actualiza la tarjeta de información.
                    var cuadro = document.getElementById(
                        'informacion-ruta'
                    );

                    if (cuadro) {

                        cuadro.innerHTML =
                            '<b style="color:#B7FF36">RUTA ' +
                            elemento.numero + ' SELECCIONADA</b><br><br>' +
                            '<b>Recorrido:</b><br>' +
                            elemento.ruta.join(' → ') + '<br><br>' +
                            '<b>Costo del grafo:</b> ' +
                            elemento.costo + ' km<br>' +
                            '<b>Distancia vial OSRM:</b> ' +
                            elemento.distancia + ' km';
                    }

                    var mensaje = document.getElementById(
                        'instruccion-ruta'
                    );

                    if (mensaje) {
                        mensaje.innerHTML =
                            '<b style="color:#B7FF36">' +
                            'ALTERNATIVA ' + elemento.numero +
                            ' SELECCIONADA</b><br>' +
                            'Puedes seleccionar otra línea.';
                    }

                });
            });

            {% endmacro %}
            """)

            macro.mapa = mapa.get_name()
            macro.rutas = lista_rutas_js
            macro.coordenadas = COORDENADAS_CIUDADES

            mapa.get_root().add_child(macro)

            print(f"Origen: {origen}")
            print(f"Destino: {destino}")
            print(f"Alternativas dibujadas: {len(capas_rutas)}")

            for elemento in capas_rutas:

                print(
                    f"Ruta {elemento['numero']}: "
                    f"{' → '.join(elemento['ruta'])} | "
                    f"Costo del grafo: {elemento['costo']} km | "
                    f"Distancia OSRM: {elemento['distancia']} km"
                )

            if errores:
                print("\nRutas que no pudieron dibujarse:")

                for error in errores:
                    print("-", error)

            print("\nHaz clic en una línea del mapa para seleccionarla.")

            display(mapa)

        except (ValueError, KeyError) as error:

            print(f"No fue posible consultar las rutas: {error}")


# ==============================================================================
# VINCULACION DE LOS BOTONES
# ==============================================================================

boton_calcular.on_click(calcular_ruta_interfaz)
boton_consultar.on_click(consultar_rutas_interfaz)


# ==============================================================================
# ESTILOS CSS - IDENTIDAD FUTURISTA NEON
# ==============================================================================

estilos_logi = widgets.HTML(value="""
<style>

/* CONTENEDOR PRINCIPAL: GRIS AZULADO, NO NEGRO */

.logi-app {
    background: #263746 !important;
    border: 1px solid #526A7C !important;
    border-radius: 16px !important;
    padding: 24px !important;
    margin: 12px 0 !important;
    color: #F3F7FA !important;
    box-shadow: 0 12px 35px rgba(20,35,50,.25) !important;
}

/* ENCABEZADO */

.logi-app .logi-header {
    background:
        linear-gradient(
            110deg,
            #00E5FF 0%,
            #B56BFF 55%,
            #FF35C8 100%
        ) top / 100% 4px no-repeat,
        #34495A;
    border: 1px solid #526A7C;
    border-radius: 12px;
    padding: 26px;
    margin-bottom: 23px;
}

.logi-app .marca {
    color: #00E5FF;
    font-size: 12px;
    letter-spacing: 5px;
    font-weight: 900;
}

.logi-app .titulo {
    color: #FFFFFF;
    font-size: 32px;
    font-weight: 900;
    margin-top: 10px;
    letter-spacing: 1px;
}

.logi-app .titulo span {
    color: #FF35C8;
    text-shadow: 0 0 12px rgba(255,53,200,.5);
}

.logi-app .subtitulo {
    color: #D0DAE2;
    font-size: 13px;
    margin-top: 8px;
}

.logi-app .estado {
    display: inline-block;
    margin-top: 18px;
    padding: 8px 13px;
    background: #40576A;
    border: 1px solid #B7FF36;
    border-radius: 20px;
    color: #B7FF36;
    font-size: 10px;
    letter-spacing: 2px;
    font-weight: 800;
}

/* TITULOS Y ETIQUETAS */

.logi-app .seccion-titulo {
    color: #00E5FF;
    font-size: 12px;
    font-weight: 900;
    letter-spacing: 2px;
    margin: 20px 0 14px;
    padding-bottom: 9px;
    border-bottom: 1px solid #597184;
}

.logi-app .campo-label {
    color: #DCE6ED;
    font-size: 11px;
    font-weight: 900;
    letter-spacing: 1px;
    margin-bottom: 8px;
}

/* SELECTORES */

.logi-app .widget-dropdown select {
    background: #40576A !important;
    color: #FFFFFF !important;
    border: 1px solid #698398 !important;
    border-left: 4px solid #00E5FF !important;
    border-radius: 8px !important;
    min-height: 42px;
    padding: 8px 11px !important;
    box-shadow: none !important;
}

.logi-app .widget-dropdown select:focus {
    border: 1px solid #FF35C8 !important;
    border-left: 4px solid #FF35C8 !important;
    outline: none !important;
    box-shadow: 0 0 12px rgba(255,53,200,.25) !important;
}

/* BOTONES */

.logi-app .widget-button button {
    border-radius: 8px !important;
    font-weight: 900 !important;
    letter-spacing: 1px;
    transition: all .2s ease;
    box-shadow: none !important;
}

.logi-app .logi-primary button {
    background: linear-gradient(
        100deg,
        #00E5FF,
        #00B8E6
    ) !important;
    color: #18303C !important;
    border: 1px solid #00E5FF !important;
    box-shadow: 0 0 13px rgba(0,229,255,.22) !important;
}

.logi-app .logi-primary button:hover {
    background: linear-gradient(
        100deg,
        #B7FF36,
        #00E5FF
    ) !important;
    color: #18303C !important;
    box-shadow: 0 0 20px rgba(0,229,255,.45) !important;
}

.logi-app .logi-secondary button {
    background: #40576A !important;
    color: #FF72D7 !important;
    border: 1px solid #FF35C8 !important;
    box-shadow: 0 0 10px rgba(255,53,200,.13) !important;
}

.logi-app .logi-secondary button:hover {
    background: #FF35C8 !important;
    color: #263746 !important;
    box-shadow: 0 0 18px rgba(255,53,200,.4) !important;
}

.logi-app .logi-add button {
    background: #40576A !important;
    color: #00E5FF !important;
    border: 1px solid #00E5FF !important;
}

.logi-app .logi-add button:hover {
    background: #00E5FF !important;
    color: #263746 !important;
}

.logi-app .boton-quitar button {
    background: #40576A !important;
    color: #FF72D7 !important;
    border: 1px solid #FF35C8 !important;
    border-radius: 7px !important;
}

/* RECORRIDO VISUAL */

.logi-app .ruta-visual {
    background: #34495A;
    border: 1px solid #526A7C;
    border-left: 4px solid #00E5FF;
    border-radius: 10px;
    padding: 19px;
    margin-top: 19px;
}

.logi-app .ruta-titulo {
    color: #00E5FF;
    font-size: 11px;
    letter-spacing: 2px;
    font-weight: 900;
    margin-bottom: 20px;
}

.logi-app .ruta-linea {
    display: flex;
    align-items: center;
    overflow-x: auto;
    padding: 4px 0 10px;
}

.logi-app .ruta-punto {
    min-width: 90px;
    text-align: center;
}

.logi-app .ruta-circulo {
    width: 34px;
    height: 34px;
    line-height: 34px;
    border-radius: 50%;
    margin: auto;
    font-size: 12px;
    font-weight: 900;
    color: #263746;
}

.logi-app .origen {
    background: #B7FF36;
    box-shadow: 0 0 15px rgba(183,255,54,.45);
}

.logi-app .parada {
    background: #B56BFF;
    color: #FFFFFF;
    box-shadow: 0 0 15px rgba(181,107,255,.4);
}

.logi-app .destino {
    background: #FF35C8;
    box-shadow: 0 0 15px rgba(255,53,200,.4);
}

.logi-app .ruta-ciudad {
    color: #FFFFFF;
    font-size: 11px;
    font-weight: 800;
    margin-top: 8px;
}

.logi-app .ruta-etiqueta {
    color: #C1CDD6;
    font-size: 9px;
    margin-top: 4px;
    letter-spacing: 1px;
}

.logi-app .ruta-conector {
    height: 3px;
    min-width: 25px;
    flex: 1;
    background: linear-gradient(
        90deg,
        #B7FF36,
        #00E5FF,
        #B56BFF,
        #FF35C8
    );
    margin-bottom: 25px;
    box-shadow: 0 0 9px rgba(0,229,255,.3);
}

/* PARADAS */

.logi-app .sin-paradas {
    background: #40576A;
    color: #D4DFE7;
    padding: 13px;
    border-radius: 8px;
    border-left: 3px solid #B56BFF;
    font-size: 11px;
}

.logi-app .parada-item {
    color: #FFFFFF;
    background: #40576A;
    border: 1px solid #597184;
    border-left: 3px solid #FF35C8;
    border-radius: 7px;
    padding: 10px 12px;
    font-size: 12px;
    width: 100%;
    margin: 4px 0;
}

.logi-app .punto-mini {
    display: inline-block;
    width: 9px;
    height: 9px;
    border-radius: 50%;
    background: #FF35C8;
    box-shadow: 0 0 8px rgba(255,53,200,.5);
    margin-right: 9px;
}

/* SEPARADORES Y PIE */

.logi-app .separador {
    height: 2px;
    background: linear-gradient(
        90deg,
        #00E5FF,
        #B56BFF,
        #FF35C8,
        transparent
    );
    margin: 23px 0;
}

.logi-app .pie {
    color: #B9C7D1;
    font-size: 10px;
    text-align: center;
    letter-spacing: 2px;
    padding: 18px 0 5px;
    border-top: 1px solid #526A7C;
    margin-top: 17px;
}

/* AREA DE RESULTADOS */

.logi-app .output_area {
    background: #34495A !important;
    color: #F3F7FA !important;
    border-radius: 9px !important;
    padding: 12px !important;
}

.logi-app .output_area pre {
    color: #F3F7FA !important;
    font-family: Arial, sans-serif !important;
    font-size: 13px !important;
    line-height: 1.7 !important;
}

/* CONTROLES GENERALES */

.logi-app .widget-label {
    color: #DCE6ED !important;
}

.logi-app .widget-dropdown {
    color: #FFFFFF !important;
}

</style>
""")


# ==============================================================================
# CONSTRUCCION DEL ENCABEZADO
# ==============================================================================

encabezado = widgets.HTML(value="""
<div class="logi-header">

    <div class="marca">SISTEMA INTELIGENTE DE LOGISTICA</div>

    <div class="titulo">
        LOGI<span>EXPRESS</span>
    </div>

    <div class="subtitulo">
        Plataforma de planificación y análisis de recorridos logísticos.
    </div>

    <div class="estado">
        ● SISTEMA DE RUTAS ACTIVO
    </div>

</div>
""")


def etiqueta(texto):
    """
    Crea una etiqueta HTML para identificar los controles.
    """

    return widgets.HTML(
        f"<div class='campo-label'>{texto}</div>"
    )


# ==============================================================================
# ORGANIZACION DE LOS CONTROLES
# ==============================================================================

tarjeta_origen = widgets.VBox([
    etiqueta("01 / PUNTO DE PARTIDA"),
    selector_origen
])

tarjeta_destino = widgets.VBox([
    etiqueta("02 / DESTINO FINAL"),
    selector_destino
])

tarjeta_paradas = widgets.VBox([

    etiqueta("03 / CIUDADES INTERMEDIAS"),

    widgets.HBox(
        [
            selector_parada,
            boton_agregar_parada
        ],
        layout=widgets.Layout(
            width="100%",
            align_items="center"
        )
    ),

    contenedor_paradas
])

bloque_acciones = widgets.VBox([

    widgets.HTML(
        "<div class='seccion-titulo'>PLANIFICACIÓN DE RECORRIDOS</div>"
    ),

    boton_calcular,
    boton_consultar

])

boton_calcular.add_class("logi-primary")
boton_consultar.add_class("logi-secondary")
boton_agregar_parada.add_class("logi-add")


# ==============================================================================
# PANEL PRINCIPAL
# ==============================================================================

panel_interfaz = widgets.VBox([

    encabezado,

    widgets.HTML(
        "<div class='seccion-titulo'>"
        "CONFIGURA TU TRAYECTORIA"
        "</div>"
    ),

    tarjeta_origen,
    tarjeta_destino,
    tarjeta_paradas,

    linea_recorrido,

    widgets.HTML("<div class='separador'></div>"),

    bloque_acciones,

    widgets.HTML(
        "<div class='pie'>"
        "LOGIEXPRESS · OPTIMIZACIÓN DE RUTAS · COLOMBIA"
        "</div>"
    ),

    salida_interfaz

])

panel_interfaz.add_class("logi-app")


# ==============================================================================
# MOSTRAR INTERFAZ
# ==============================================================================

display(estilos_logi)
display(panel_interfaz)


HTML(value='\n<style>\n\n/* CONTENEDOR PRINCIPAL: GRIS AZULADO, NO NEGRO */\n\n.logi-app {\n    background: #2…